# 1주차. 모듈, 패키지, API 스타일 ETL

| 항목 | 내용 |
|---|---|
| 이름 | 이성민 |
| 학과 | 소프트웨어융합과 |
| 학년 | 2학년 |
| 학번 | 2151050 |

이 노트북은 수업 주차 흐름을 참고해 우리 방식으로 재구성한 실행형 설명 자료입니다. 외부 서비스 호출 없이 실행되도록 작은 샘플 데이터를 사용합니다.

## 핵심 아이디어
모듈은 함수를 파일 단위로 묶는 방식이고, 패키지는 여러 모듈을 폴더 구조로 묶는 방식입니다.
여기서는 API 응답처럼 생긴 데이터를 가져와 정리하는 ETL 흐름을 함수 단위로 나눕니다.

In [1]:
import pandas as pd

def extract_launches():
    return [
        {"mission": "KBU-1", "year": 2024, "payload": 350, "success": True},
        {"mission": "KBU-2", "year": 2025, "payload": 720, "success": True},
        {"mission": "KBU-3", "year": 2026, "payload": 510, "success": False},
    ]

def transform_launches(rows):
    df = pd.DataFrame(rows)
    df["success_flag"] = df["success"].astype(int)
    df["payload_ton"] = df["payload"] / 1000
    return df

def load_summary(df):
    return {
        "missions": len(df),
        "success_rate": round(df["success_flag"].mean(), 2),
        "total_payload_ton": round(df["payload_ton"].sum(), 2),
    }

launches = transform_launches(extract_launches())
summary = load_summary(launches)

assert summary["missions"] == 3
assert 0 <= summary["success_rate"] <= 1
display(launches)
summary

,mission,year,payload,success,success_flag,payload_ton
0,KBU-1,2024,350,True,1,0.35
1,KBU-2,2025,720,True,1,0.72
2,KBU-3,2026,510,False,0,0.51


{'missions': 3,
 'success_rate': np.float64(0.67),
 'total_payload_ton': np.float64(1.58)}

## Titanic 스타일 전처리
같은 ETL 구조를 승객 데이터에도 적용합니다. 결측값을 먼저 처리해야 그룹별 생존율 계산이 안정적으로 동작합니다.

In [2]:
passengers = pd.DataFrame([
    {"pclass": 1, "sex": "female", "age": 38, "survived": 1},
    {"pclass": 3, "sex": "male", "age": 22, "survived": 0},
    {"pclass": 2, "sex": "female", "age": None, "survived": 1},
    {"pclass": 3, "sex": "male", "age": 35, "survived": 0},
])

passengers["age"] = passengers["age"].fillna(passengers["age"].median())
survival = passengers.groupby(["pclass", "sex"], as_index=False)["survived"].mean()

assert passengers["age"].isna().sum() == 0
display(survival)

,pclass,sex,survived
0,1,female,1.0
1,2,female,1.0
2,3,male,0.0
